# Silver Exploration Notebook


## Imports

In [1]:
import os
import findspark

from pyspark.sql import SparkSession

from pyspark.sql.functions import (
    col,
    lit,
    when,
    to_timestamp,
    from_utc_timestamp,
    dayofweek,
    hour,
    month,
    input_file_name,
    regexp_extract,
    arrays_zip,
    explode,
    size,
    count,
    length,
    min,
    max,
    avg,
    slice,
    regexp_extract,
    desc,
    unix_micros,
    abs,
    split,
    element_at,
    sum as spark_sum
)

## Start Spark

In [2]:
findspark.init()

spark = (
    SparkSession.builder
    .appName("silver-exploration")
    .config("spark.sql.session.timeZone", "UTC")
    .getOrCreate()
)

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/11/10 12:58:00 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [3]:
RAW = os.path.expanduser("~/spark-lab-data/dev/project/raw")
print(RAW)

/home/ubuntu/spark-lab-data/dev/project/raw


## Read Bike-sharing dataset
+ Adding city and month

In [4]:
# Bike-sharing data
trips_raw = (
    spark.read
    .option("header", True)
    .csv(f"{RAW}/trips/*/*.csv")
    .withColumn("src", input_file_name())
)

trips_raw = (
    trips_raw
    .withColumn("city", element_at(split(col("src"), "/"), -2))
    .withColumn("year_month", element_at(split(col("src"), "/"), -1))
    .withColumn("year_month",split(col("year_month"), r"\.").getItem(0))
)

trips_raw.select("city", "year_month").show(5)

+----+----------+
|city|year_month|
+----+----------+
|oslo|   2022-08|
|oslo|   2022-08|
|oslo|   2022-08|
|oslo|   2022-08|
|oslo|   2022-08|
+----+----------+
only showing top 5 rows



## Exporation of Bike-sharing dataset

In [5]:
trips_raw.printSchema()
trips_raw.show(5, truncate=False)
print("Total rows:", trips_raw.count())
print("Columns:", trips_raw.columns)

root
 |-- started_at: string (nullable = true)
 |-- ended_at: string (nullable = true)
 |-- duration: string (nullable = true)
 |-- start_station_id: string (nullable = true)
 |-- start_station_name: string (nullable = true)
 |-- start_station_description: string (nullable = true)
 |-- start_station_latitude: string (nullable = true)
 |-- start_station_longitude: string (nullable = true)
 |-- end_station_id: string (nullable = true)
 |-- end_station_name: string (nullable = true)
 |-- end_station_description: string (nullable = true)
 |-- end_station_latitude: string (nullable = true)
 |-- end_station_longitude: string (nullable = true)
 |-- src: string (nullable = false)
 |-- city: string (nullable = true)
 |-- year_month: string (nullable = true)

+--------------------------------+--------------------------------+--------+----------------+-------------------------+----------------------------+----------------------+-----------------------+--------------+------------------+-----------

Total rows: 8460580
Columns: ['started_at', 'ended_at', 'duration', 'start_station_id', 'start_station_name', 'start_station_description', 'start_station_latitude', 'start_station_longitude', 'end_station_id', 'end_station_name', 'end_station_description', 'end_station_latitude', 'end_station_longitude', 'src', 'city', 'year_month']


- We have a total of 8 428 152 biketrips in Oslo, Trondheim and Bergen.
- The columns in the Bike-sharing dataset include: `started_at`, `endend_at`, `duration`, `start_station_id`, `start_station_name`, `start_station_description`, `start_station_latitude`, `start_station_longitude`, `end_station_id`, `end_station_name`, `end_station_description`, `end_station_latitude`, `end_station_longitude`. We also have the added `src`, `city` and `year_month`.

#### Data Types
The 16 columns are stored as string data types. This is because PySpark reads CSV columns as strings by default. However, Several columns represent numeric values or timestamps. We will therefore convert these columns to appropriate data types. 

- Timestamps: `started_at`and `ended_at` -> timestamp
- Trip duration: `duration` -> integer(maybe double?)
- Coorinates: `latitude`and `longitude`-> double

Maybe:
- StationIDs: `start_station_id` and `stop_station_id` -> keep as string og convert to integer

From the table above we can see the ID is stored as a numeric value, but we would argue to keep them as a string as they are identifiers rather than numeric measurements. This also provides a greater flexibility if future datasets contains letters(e.g. ST123), without requiring changes to the data type. 


### Investigating missing values

In [6]:
trips_raw.select([
    spark_sum(col(c).isNull().cast("int")).alias(c)
    for c in trips_raw.columns
]).show()

+----------+--------+--------+----------------+------------------+-------------------------+----------------------+-----------------------+--------------+----------------+-----------------------+--------------------+---------------------+---+----+----------+
|started_at|ended_at|duration|start_station_id|start_station_name|start_station_description|start_station_latitude|start_station_longitude|end_station_id|end_station_name|end_station_description|end_station_latitude|end_station_longitude|src|city|year_month|
+----------+--------+--------+----------------+------------------+-------------------------+----------------------+-----------------------+--------------+----------------+-----------------------+--------------------+---------------------+---+----+----------+
|         0|       0|       0|               0|                 0|                    11126|                     0|                      0|             0|               0|                   7071|                   0|       

#### Missing values
The `start_station_description` column contains 11 000 missing values, while `end_station_description` contains 6950 missing values.

#### Is this detrimental to our project or analysis?
These columns are not truly important for the type of analysing we want to do. Our main objective is to investigate how weather conditions influence bike-sharing usage. Since station ID and names are available, the missing descriptions do not prevent us from identifying stations and analyzing trips. 

Although they are not required for our current objectives, they could be valuable for other purposes later. In the Gold layer, we can simply exclude columns that are not relevant to are final analytical dataset. 

### Investigating duplicates

In [7]:
print("Total rows:", trips_raw.count())
print("Duplicate rows:", trips_raw.count() - trips_raw.distinct().count())

Total rows: 8460580


26/11/10 12:58:35 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/11/10 12:58:35 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/11/10 12:58:35 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/11/10 12:58:36 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/11/10 12:58:36 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/11/10 12:58:36 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/11/10 12:58:36 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/11/10 12:58:36 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/11/10 12:58:38 WARN RowBasedKeyValueBatch: Calling spill() on

Duplicate rows: 25


Out of the 8 428 152 trips, 25 additional duplicates. 

#### Are we sure these are duplicates?

We can not be certain that these are dublicates because the trips do not contain any identifier. These records do however contain identical entries across all 16 collumns, including timestamp with microseconds precision. It is highly unlikely that separate trips would share all these values.

We consider therefore these records to be duplicate entries, potentially from the source data or overlapping records during the ingestion process. The exact cause has not been investigated further, but we want to remove the additional copies. 

Given that our dataset contain over 8 million trips, removing 25 duplicate records will have a negligible impact on the overall analysis. If the dataset was significanlty smaller, further investigation would be important as removing even a small number of records could have a greater influence on the result.

In [8]:
trips_silver = trips_raw.dropDuplicates()

### Investigatin values and distributions

#### First Duration
Lets start with convert the `duration` column to a double data type. 

In [9]:
trips_silver = (
    trips_silver
    .withColumn("duration", col("duration").cast("double"))
)

# Did the convertion work?
trips_silver.filter(
    col("duration").isNull() 
).count()


0

In [10]:

trips_silver.select("duration").summary("min", "25%", "50%", "75%", "max").show()

trips_silver.select(spark_sum(when(col("duration") <= 0, 1).otherwise(0)).alias("non_positive"),
         spark_sum(when(col("duration") < 60, 1).otherwise(0)).alias("under_60s"),
         spark_sum(when(col("duration").isNull(), 1).otherwise(0)).alias("null_duration")).show()

26/11/10 12:59:36 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/11/10 12:59:36 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/11/10 12:59:36 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/11/10 12:59:36 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/11/10 12:59:36 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/11/10 12:59:36 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/11/10 12:59:36 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/11/10 12:59:36 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/11/10 12:59:38 WARN RowBasedKeyValueBatch: Calling spill() on

+-------+--------+
|summary|duration|
+-------+--------+
|    min|     0.0|
|    25%|   340.0|
|    50%|   524.0|
|    75%|   805.0|
|    max| 79790.0|
+-------+--------+



26/11/10 13:00:24 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/11/10 13:00:24 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/11/10 13:00:24 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/11/10 13:00:24 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/11/10 13:00:24 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/11/10 13:00:24 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/11/10 13:00:24 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/11/10 13:00:24 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/11/10 13:00:25 WARN RowBasedKeyValueBatch: Calling spill() on

+------------+---------+-------------+
|non_positive|under_60s|null_duration|
+------------+---------+-------------+
|           2|     1304|            0|
+------------+---------+-------------+



#### Durations to investigate ferther
- 2 trips has a negative or a duration of 0.
- Is it possible that a trip last around 22 hours? (79790/3600 = 22)

In [11]:
# Trips with durations smaller than 1 second
trips_silver.filter(
    col("duration").cast("double") <= 1
).show(truncate=False)

print("How many trips have a duration of 1 second or less?", trips_silver.filter(col("duration").cast("double") <= 1).count())

+--------------------------------+--------------------------------+--------+----------------+------------------------+-------------------------------------------------------+----------------------+-----------------------+--------------+-----------------------+------------------------------------------------------------------+--------------------+---------------------+------------------------------------------------------------------------------+---------+----------+
|started_at                      |ended_at                        |duration|start_station_id|start_station_name      |start_station_description                              |start_station_latitude|start_station_longitude|end_station_id|end_station_name       |end_station_description                                           |end_station_latitude|end_station_longitude|src                                                                           |city     |year_month|
+--------------------------------+------------------------

How many trips have a duration of 1 second or less? 29


### Extremely Short Trips
29 trips with a recorded duration of one second or less, including two trips with a duration of zero seconds. 

With the table above these trips have a different start and end station which makes it physically impossible to complete the journeys within that timeframe.

These records likely represent data quality issues that need futher investigation.

Possible solution: 
- Find the mean trip duration from one statiotn to the other and compare it to minimum duration record.
- Flag trips under a spesific timeframe. 


In [12]:
station_durations = (
    trips_silver
    .groupBy("start_station_id", "end_station_id")
    .agg(
        count("*").alias("num_trips"),
        avg("duration").alias("mean_duration"),
        min("duration").alias("min_duration")
    )
)

station_durations.filter(
    col("num_trips") >= 10
).orderBy("min_duration").show(20, truncate=False)

26/11/10 13:01:32 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/11/10 13:01:32 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/11/10 13:01:32 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/11/10 13:01:32 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/11/10 13:01:32 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/11/10 13:01:32 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/11/10 13:01:32 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/11/10 13:01:32 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/11/10 13:01:34 WARN RowBasedKeyValueBatch: Calling spill() on

+----------------+--------------+---------+------------------+------------+
|start_station_id|end_station_id|num_trips|mean_duration     |min_duration|
+----------------+--------------+---------+------------------+------------+
|465             |523           |93       |585.3978494623656 |0.0         |
|217             |815           |373      |241.01072386058982|0.0         |
|819             |221           |170      |1180.2529411764706|1.0         |
|44              |123           |502      |513.4681274900398 |1.0         |
|25              |1             |92       |1408.391304347826 |1.0         |
|10              |293           |200      |274.615           |1.0         |
|212             |83            |218      |457.76605504587155|1.0         |
|155             |1882          |303      |720.1221122112212 |1.0         |
|788             |808           |1001     |409.4725274725275 |1.0         |
|105             |108           |251      |453.43824701195217|1.0         |
|14         

### What does the table above tell us?
Although the mean duration does not accually tell us the minimum possible duration between two stations, it provides a useful indication of what a typical trip looks like. 

The table shows station pairs with hundres and thousands of bike-trips, where the average duration is several minutes, yet the shortest recorded time is only one second. This strongly suggest that some values are incorrect or unrealistic. 

On another note this does make me question the dataset and its accuracy. Are these simply a small number of isolated data quality issues, or could they indicate more problems with the data?

For now let's atleast flag trips with a duration smaller than one second. 

In [13]:
# Flag trips that are under 1 second
trips_silver = trips_silver.withColumn(
    "short_trip_flag",
    col("duration") <= 1
)

### Trips with a long duration

In [15]:
print("Number of trips longer than 12 hours: ", trips_silver.filter(col("duration") > 12 * 3600).count())

trips_silver.select(
    "started_at",
    "ended_at",
    "duration"
).orderBy(desc("duration")).show(15, truncate=False)

Number of trips longer than 12 hours:  15


26/11/10 13:02:48 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/11/10 13:02:48 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/11/10 13:02:49 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/11/10 13:02:49 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/11/10 13:02:49 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/11/10 13:02:49 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/11/10 13:02:49 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/11/10 13:02:49 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/11/10 13:02:50 WARN RowBasedKeyValueBatch: Calling spill() on

+--------------------------------+--------------------------------+--------+
|started_at                      |ended_at                        |duration|
+--------------------------------+--------------------------------+--------+
|2026-06-19 13:38:13.233000+00:00|2026-06-20 11:48:03.076000+00:00|79790.0 |
|2025-10-17 14:35:20.232000+00:00|2025-10-18 11:15:52.091000+00:00|74431.0 |
|2022-04-16 12:47:44.363000+00:00|2022-04-17 07:37:10.178000+00:00|67765.0 |
|2023-04-07 13:40:11.612000+00:00|2023-04-08 08:18:37.147000+00:00|67105.0 |
|2023-05-31 15:31:03.704000+00:00|2023-06-01 08:37:01.096000+00:00|61557.0 |
|2026-05-04 18:28:16.356000+00:00|2026-05-05 11:14:23.840000+00:00|60367.0 |
|2022-11-01 15:19:05.897000+00:00|2022-11-02 07:26:44.435000+00:00|58058.0 |
|2022-11-24 14:20:12.725000+00:00|2022-11-25 05:44:40.784000+00:00|55468.0 |
|2026-08-06 20:38:28.625000+00:00|2026-08-07 11:37:01.191000+00:00|53913.0 |
|2022-05-20 16:49:31.692000+00:00|2022-05-21 06:44:45.835000+00:00|50114.0 |

Could a city bike realistically be used for appocimately 22 hours?

Accoring to Oslo Bysykkel, a fully charged electric bike has an estimate range of around 70 kilometers. The battery does not necessarily limit the recorded trip duration, as the bike can remain parked while the rental is still active and you can manually ride the bike if the battery runs out. 

If the dataset included distance travelled we could better investigate whether the bicycle was actively used.

Possible explainations for the long duration
- The user forgot to return the bike or didn't properly complete the return process.
- The bike was parked for several hours while the rental was still active. 
- The recorded timestamp contains an error. 
- Maybe others?

In [16]:
print("Over 3 hours:", trips_silver.filter(col("duration") > 3 * 60 * 60).count())
print("Over 6 hours:", trips_silver.filter(col("duration") > 6 * 60 * 60).count())
print("Over 12 hours:", trips_silver.filter(col("duration") > 12 * 60 * 60).count())

Over 3 hours: 3184


Over 6 hours: 315


Over 12 hours: 15


(3184/8460580) * 100 = 0.0376 %

Only 0.038% of all bike trips last longer than 3 hours, inducating such trips are extremely rare.

### Investigating the timestamp format

In [17]:
trips_silver.groupBy(length("started_at").alias("len")).count().orderBy("len").show()
trips_silver.select("started_at", "ended_at", "duration").distinct().limit(10).show(truncate=False)

26/11/10 13:10:14 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/11/10 13:10:14 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/11/10 13:10:14 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/11/10 13:10:14 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/11/10 13:10:14 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/11/10 13:10:15 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/11/10 13:10:15 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/11/10 13:10:15 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/11/10 13:10:16 WARN RowBasedKeyValueBatch: Calling spill() on

+---+-------+
|len|  count|
+---+-------+
| 25|   8219|
| 32|8452336|
+---+-------+



26/11/10 13:11:04 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/11/10 13:11:07 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.


+--------------------------------+--------------------------------+--------+
|started_at                      |ended_at                        |duration|
+--------------------------------+--------------------------------+--------+
|2022-08-01 05:35:31.981000+00:00|2022-08-01 05:51:32.968000+00:00|960.0   |
|2022-08-01 05:39:23.445000+00:00|2022-08-01 05:54:46.326000+00:00|922.0   |
|2022-08-01 07:28:41.156000+00:00|2022-08-01 07:40:31.128000+00:00|709.0   |
|2022-08-01 07:33:41.640000+00:00|2022-08-01 07:43:22.460000+00:00|580.0   |
|2022-08-01 09:31:48.451000+00:00|2022-08-01 09:41:56.865000+00:00|608.0   |
|2022-08-01 10:25:16.864000+00:00|2022-08-01 10:29:04.710000+00:00|227.0   |
|2022-08-01 10:34:29.583000+00:00|2022-08-01 10:49:44.016000+00:00|914.0   |
|2022-08-01 13:36:38.583000+00:00|2022-08-01 13:43:00.941000+00:00|382.0   |
|2022-08-01 14:31:37.484000+00:00|2022-08-01 14:40:59.374000+00:00|561.0   |
|2022-08-01 14:35:07.091000+00:00|2022-08-01 14:45:50.809000+00:00|643.0   |

#### Converting the string to UTC timestamp
The `started_at` and `ended_at` timestamps are recorded in UTC, as indicated by `+00:00`.

For example:
- `started_at`: 2022-08-01 05:14:13.781000+00:00
- `ended_at`: 2022-08-01 05:26:27.585000+00:00

In [18]:
# Setting the spark session timezone to UTC
spark.conf.set("spark.sql.session.timeZone", "UTC")

# Converting the started_at and ended_at columns to timestamp type
trips_silver = (
    trips_silver
    .withColumn("started_ts", to_timestamp(col("started_at")))
    .withColumn("ended_ts", to_timestamp(col("ended_at")))
)

# Let's compare the converted timestamps
trips_silver.select(
    "started_at",
    "started_ts",
    "ended_at",
    "ended_ts"
).show(10, truncate=False)

26/11/10 13:11:54 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/11/10 13:11:54 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/11/10 13:11:54 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/11/10 13:11:54 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/11/10 13:11:54 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/11/10 13:11:54 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/11/10 13:11:54 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/11/10 13:11:54 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/11/10 13:11:55 WARN RowBasedKeyValueBatch: Calling spill() on

+--------------------------------+-----------------------+--------------------------------+-----------------------+
|started_at                      |started_ts             |ended_at                        |ended_ts               |
+--------------------------------+-----------------------+--------------------------------+-----------------------+
|2022-08-01 07:27:12.318000+00:00|2022-08-01 07:27:12.318|2022-08-01 07:33:16.030000+00:00|2022-08-01 07:33:16.03 |
|2022-08-01 07:44:25.763000+00:00|2022-08-01 07:44:25.763|2022-08-01 07:56:09.403000+00:00|2022-08-01 07:56:09.403|
|2022-08-01 11:40:29.713000+00:00|2022-08-01 11:40:29.713|2022-08-01 11:53:28.469000+00:00|2022-08-01 11:53:28.469|
|2022-08-01 12:15:03.216000+00:00|2022-08-01 12:15:03.216|2022-08-01 12:25:57.070000+00:00|2022-08-01 12:25:57.07 |
|2022-08-01 12:57:22.639000+00:00|2022-08-01 12:57:22.639|2022-08-01 13:07:39.778000+00:00|2022-08-01 13:07:39.778|
|2022-08-01 13:08:46.988000+00:00|2022-08-01 13:08:46.988|2022-08-01 13:

In [19]:
# Did the convertion work?
trips_silver.filter(
    col("started_ts").isNull() |
    col("ended_ts").isNull()
).count()

0

### Converting Timestamps to Norwegian Local Time

We convert these timestamps to Norwegian local time (`Europe/Oslo`), accounting for daylight saving time. The original UTC timestamps are preserved for further processing.

In [20]:
trips_silver = (
    trips_silver
    .withColumn(
        "started_local",
        from_utc_timestamp(
            to_timestamp(col("started_ts")),
            "Europe/Oslo"
        )
    )
    .withColumn(
        "ended_local",
        from_utc_timestamp(
            to_timestamp(col("ended_ts")),
            "Europe/Oslo"
        )
    )
)

trips_silver.select("started_ts","started_local","ended_ts","ended_local").show(10, truncate=False)

26/11/10 13:13:07 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/11/10 13:13:07 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/11/10 13:13:07 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/11/10 13:13:07 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/11/10 13:13:07 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/11/10 13:13:07 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/11/10 13:13:07 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/11/10 13:13:07 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/11/10 13:13:09 WARN RowBasedKeyValueBatch: Calling spill() on

+-----------------------+-----------------------+-----------------------+-----------------------+
|started_ts             |started_local          |ended_ts               |ended_local            |
+-----------------------+-----------------------+-----------------------+-----------------------+
|2022-08-01 07:27:12.318|2022-08-01 09:27:12.318|2022-08-01 07:33:16.03 |2022-08-01 09:33:16.03 |
|2022-08-01 07:44:25.763|2022-08-01 09:44:25.763|2022-08-01 07:56:09.403|2022-08-01 09:56:09.403|
|2022-08-01 11:40:29.713|2022-08-01 13:40:29.713|2022-08-01 11:53:28.469|2022-08-01 13:53:28.469|
|2022-08-01 12:15:03.216|2022-08-01 14:15:03.216|2022-08-01 12:25:57.07 |2022-08-01 14:25:57.07 |
|2022-08-01 12:57:22.639|2022-08-01 14:57:22.639|2022-08-01 13:07:39.778|2022-08-01 15:07:39.778|
|2022-08-01 13:08:46.988|2022-08-01 15:08:46.988|2022-08-01 13:25:32.594|2022-08-01 15:25:32.594|
|2022-08-01 14:12:35.452|2022-08-01 16:12:35.452|2022-08-01 14:19:11.994|2022-08-01 16:19:11.994|
|2022-08-01 14:24:32

#### Does duration correspond to the start and end time?

In [24]:
trips_silver.select(
    "started_local",
    "ended_local",
    "duration",
    # does started_at and ended_at match the duration?
    (
        (
            unix_micros(to_timestamp("ended_local")) -
            unix_micros(to_timestamp("started_local"))
        ) / 1000000
    ).alias("duration_check")

).show(5, truncate=False)

26/11/10 13:22:01 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/11/10 13:22:01 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/11/10 13:22:01 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/11/10 13:22:01 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/11/10 13:22:01 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/11/10 13:22:01 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/11/10 13:22:01 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/11/10 13:22:01 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/11/10 13:22:02 WARN RowBasedKeyValueBatch: Calling spill() on

+-----------------------+-----------------------+--------+--------------+
|started_local          |ended_local            |duration|duration_check|
+-----------------------+-----------------------+--------+--------------+
|2022-08-01 09:27:12.318|2022-08-01 09:33:16.03 |363.0   |363.712       |
|2022-08-01 09:44:25.763|2022-08-01 09:56:09.403|703.0   |703.64        |
|2022-08-01 13:40:29.713|2022-08-01 13:53:28.469|778.0   |778.756       |
|2022-08-01 14:15:03.216|2022-08-01 14:25:57.07 |653.0   |653.854       |
|2022-08-01 14:57:22.639|2022-08-01 15:07:39.778|617.0   |617.139       |
+-----------------------+-----------------------+--------+--------------+
only showing top 5 rows




From the table above it looks like the duration always round down.

for example:
- 363 and 363.712
- 703 and 703
- 778 and 778.756

Let's investigate the duration a bit further.

In [25]:
trips_check = trips_silver.withColumn(
    "duration_diff",
    abs(
        (unix_micros(to_timestamp("ended_local"))
         - unix_micros(to_timestamp("started_local"))) / 1_000_000
        - col("duration").cast("double")
    )
)

trips_check.select("duration_diff").summary().show()

26/11/10 13:26:14 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/11/10 13:26:14 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/11/10 13:26:15 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/11/10 13:26:15 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/11/10 13:26:15 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/11/10 13:26:15 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/11/10 13:26:15 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/11/10 13:26:15 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/11/10 13:26:16 WARN RowBasedKeyValueBatch: Calling spill() on

+-------+-------------------+
|summary|      duration_diff|
+-------+-------------------+
|  count|            8460555|
|   mean|0.45581169131339366|
| stddev| 0.2852514191823245|
|    min|                0.0|
|    25%|0.21199999999998909|
|    50%|0.42500000000001137|
|    75%|  0.696999999999889|
|    max| 0.9990000000000236|
+-------+-------------------+



Since the maximum difference are below one second, we consider the recorded durations consistent with the timestamps. 

### Investigating latitude and longitude
first converting the data types to double

In [27]:
trips_silver = (
    trips_silver
    .withColumn("start_station_latitude", col("start_station_latitude").cast("double"))
    .withColumn("start_station_longitude", col("start_station_longitude").cast("double"))
    .withColumn("end_station_latitude", col("end_station_latitude").cast("double"))
    .withColumn("end_station_longitude", col("end_station_longitude").cast("double"))
)

# Did the convertion work?
trips_silver.filter(
    col("start_station_latitude").isNull() |
    col("start_station_longitude").isNull() |
    col("end_station_latitude").isNull() |
    col("end_station_longitude").isNull()
).count()

0

#### Valid cordination for latitude and longitude in Norway:
- Norwegian latitude: 57 - 71
- Norwegian longitude: 4 - 31

In [29]:
trips_silver.filter(
    (col("start_station_latitude") < 57) |
    (col("start_station_latitude") > 71) |
    (col("end_station_latitude") < 57) |
    (col("end_station_latitude") > 71) |
    (col("start_station_longitude") < 4) |
    (col("start_station_longitude") > 31) |
    (col("end_station_longitude") < 4) |
    (col("end_station_longitude") > 31)
).count()

0

In [30]:
trips_silver.select(
    min("start_station_latitude").alias("min_lat"),
    max("start_station_latitude").alias("max_lat"),
    min("start_station_longitude").alias("min_lon"),
    max("start_station_longitude").alias("max_lon")
).show()

+-----------------+------------------+-----------------+-----------------+
|          min_lat|           max_lat|          min_lon|          max_lon|
+-----------------+------------------+-----------------+-----------------+
|59.89843428245928|63.446059058709096|5.273174034250616|10.81431433498766|
+-----------------+------------------+-----------------+-----------------+



The latitude and logitude values appear to reasonable based on the geographical locations of the three cities. Oslo is the city most southern, while Trondheim is the city most nothern. Bergen is the city most western, while oslo is the city most eastern. 

|   City       |   latitude    |   longitude   |
|--------------|---------------|---------------|
|  Oslo:       |     59.91     |     10.75     |  
|  Bergen:     |     60.39     |      5.32     |
|  Trondheim:  |     63.43     |     10.40     |

### Number of trips divided by the different cities

In [ ]:
trips_silver = trips_silver.withColumn(
    "src", input_file_name()
)

trips_silver = trips_silver.withColumn(
    "city", element_at(split(col("src"), "/"), -2)
)

trips_silver = trips_silver.withColumn(
    "year_month",
    split(element_at(split(col("src"), "/"), -1), r"\.").getItem(0)
)

trips_silver.select("city", "year_month").show(5)

trips_silver.groupBy("city").count().show()

+---------+-------+
|     city|  count|
+---------+-------+
|     oslo|5718138|
|   bergen|1904236|
|trondheim| 805778|
+---------+-------+

